    # Notebook 1 — Résoudre le problème de Markowitz

    **Le Quotient Financier · Travaux pratiques d'histoire de la gestion de portefeuille**

    En 1952, Harry Markowitz pose le problème : comment combiner des titres pour obtenir le moins de risque
    possible pour un rendement visé ? Ce notebook le résout de bout en bout, sur dix vraies actions américaines.

    **Au programme**

    1. Récupérer les données et calculer rendements, variances et covariances.
    2. Affronter « l'éléphant » : l'estimation des rendements espérés.
    3. Résoudre le problème pas à pas : deux titres, puis la formule fermée en matrices.
    4. Le résoudre numériquement, avec et sans vente à découvert.
    5. Mesurer le coût du calcul quand le nombre de titres augmente.
    6. Tester la sensibilité de la solution : l'optimiseur, « maximiseur d'erreurs ».

    **Prérequis** : Python, `numpy`, `pandas`, notions de base de statistique (moyenne, variance, covariance)
    et de calcul matriciel (produit, transposée, inverse).

    **Mode d'emploi** : exécutez les cellules dans l'ordre. Dans la version étudiant, les cellules marquées
    `À COMPLÉTER` doivent être remplies avant de passer à la suite.

> *Ce notebook est un support pédagogique du Quotient Financier. Il illustre des méthodes historiques
> et académiques ; il ne constitue en aucun cas un conseil en investissement ni une recommandation
> d'achat ou de vente de titres. Les titres utilisés servent uniquement d'exemples.*

## 0. Mise en place

In [ ]:
# Sur Google Colab ou une installation neuve, décommentez la ligne suivante :
# !pip install yfinance numpy pandas scipy matplotlib

import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

import logging
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)   # masque les avertissements de polices
plt.rcParams.update({"figure.figsize": (9, 5), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "mathtext.fontset": "dejavusans"})   # formules des axes (ex. 10^1) avec la police par défaut
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
np.set_printoptions(precision=4, suppress=True)

SEMAINES = 52              # nombre de semaines par an, pour annualiser
TAUX_SANS_RISQUE = 0.02    # taux sans risque annuel retenu (hypothèse simplificatrice)

In [ ]:
# Dix titres américains, cinq secteurs, deux titres par secteur
TITRES = {
    "AAPL": "Technologie", "MSFT": "Technologie",
    "JPM": "Banque",       "BAC": "Banque",
    "XOM": "Énergie",      "CVX": "Énergie",
    "JNJ": "Santé",        "PFE": "Santé",
    "KO": "Consommation",  "PG": "Consommation",
}
MARCHE = "SPY"   # ETF S&P 500, utilisé comme indice de marché
ETF_SECTEURS = {"XLK": "Technologie", "XLF": "Banque", "XLE": "Énergie",
                "XLV": "Santé", "XLP": "Consommation"}

SECTEURS = {**TITRES, **ETF_SECTEURS, MARCHE: "Marché"}
TOUS = list(TITRES) + [MARCHE] + list(ETF_SECTEURS)
DEBUT, FIN = "2015-01-01", "2024-12-31"
FICHIER = "donnees_prix.csv"

In [ ]:
def donnees_simulees(tickers, secteurs, debut, fin, graine=42):
    """Prix hebdomadaires simulés par un modèle à facteurs (marché + secteurs + bruit à queues épaisses).
    Utilisés uniquement si le téléchargement échoue, pour que le notebook reste exécutable."""
    rng = np.random.default_rng(graine)
    dates = pd.date_range(debut, fin, freq="W-FRI")
    T = len(dates)
    t5 = lambda: rng.standard_t(5, T) / np.sqrt(5 / 3)          # loi de Student, variance 1
    marche = 0.0025 + 0.022 * t5()
    facteurs = {s: 0.015 * t5() for s in set(secteurs.values())}
    rend = {}
    for t in tickers:
        s = secteurs.get(t, "Marché")
        if t == "SPY":
            r = marche + 0.001 * rng.standard_normal(T)
        elif t == "AGG":
            r = 0.0006 + 0.008 * rng.standard_normal(T) - 0.05 * marche
        elif t.startswith("XL"):
            r = rng.uniform(0.8, 1.2) * marche + facteurs[s] + 0.003 * rng.standard_normal(T)
        else:
            r = (0.0006 * rng.standard_normal() + rng.uniform(0.6, 1.4) * marche
                 + rng.uniform(0.8, 1.5) * facteurs[s] + 0.018 * t5())
        rend[t] = r
    return 100 * (1 + pd.DataFrame(rend, index=dates)).cumprod()


def charger_prix(tickers, secteurs, debut, fin, fichier):
    """Lit le fichier local s'il existe, sinon télécharge sur Yahoo Finance,
    sinon (pas d'accès internet) bascule sur des données simulées."""
    if os.path.exists(fichier):
        prix = pd.read_csv(fichier, index_col=0, parse_dates=True)
        if set(tickers) <= set(prix.columns):
            print(f"Données lues dans {fichier}.")
            return prix[tickers], "fichier local"
    try:
        import yfinance as yf
        prix = yf.download(tickers, start=debut, end=fin, auto_adjust=True, progress=False)["Close"]
        if prix is None or len(prix) == 0 or prix.dropna(how="all").empty:
            raise ValueError("aucune donnée reçue")
        prix = prix[tickers].dropna(how="all")
        manquants = [t for t in tickers if prix[t].isna().mean() > 0.05]
        if prix.empty or manquants:
            raise ValueError(f"séries incomplètes : {manquants}")
        prix.to_csv(fichier)
        print(f"Données téléchargées sur Yahoo Finance et enregistrées dans {fichier}.")
        return prix, "Yahoo Finance"
    except Exception as e:
        print("⚠️  Téléchargement impossible (" + str(e)[:90] + ").")
        print("⚠️  LE NOTEBOOK UTILISE DES DONNÉES SIMULÉES : les chiffres ne décrivent pas les vrais titres.")
        return donnees_simulees(tickers, secteurs, debut, fin), "simulées"


def rendements_hebdo(prix):
    """Rendements hebdomadaires simples, calculés sur les cours du vendredi."""
    return prix.resample("W-FRI").last().pct_change(fill_method=None).dropna(how="any")


def boites(ax, donnees, noms):
    """Boîtes à moustaches, une par colonne de `donnees` (compatible avec toutes les versions de matplotlib)."""
    ax.boxplot(np.asarray(donnees), showfliers=False)
    ax.set_xticks(range(1, len(noms) + 1))
    ax.set_xticklabels(list(noms), rotation=90)


def afficher_matrice(M, titre, vmin=-1, vmax=1, cmap="RdBu_r", fmt="{:.2f}"):
    """Affiche une matrice (DataFrame carré) sous forme de carte de chaleur annotée."""
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(M.values, vmin=vmin, vmax=vmax, cmap=cmap)
    ax.set_xticks(range(len(M))); ax.set_xticklabels(M.columns, rotation=90)
    ax.set_yticks(range(len(M))); ax.set_yticklabels(M.index)
    for i in range(len(M)):
        for j in range(len(M)):
            ax.text(j, i, fmt.format(M.values[i, j]), ha="center", va="center", fontsize=7)
    ax.grid(False); ax.set_title(titre)
    fig.colorbar(im, ax=ax, shrink=0.8)
    plt.show()

## 1. Les données

On télécharge dix ans de cours ajustés (dividendes et divisions d'actions inclus) pour nos dix titres,
l'ETF S&P 500 (`SPY`) et cinq ETF sectoriels, qui serviront dans les notebooks suivants.
Les données sont enregistrées dans `donnees_prix.csv` : les notebooks 2 et 3 les réutilisent.

Dans ce notebook, on estime les paramètres sur les **cinq dernières années** (2020-2024),
en rendements **hebdomadaires**, soit environ 260 observations par titre.

In [ ]:
prix, SOURCE = charger_prix(TOUS, SECTEURS, DEBUT, FIN, FICHIER)
rendements = rendements_hebdo(prix)
rend = rendements[list(TITRES)].loc["2020":"2024"]
N, T = rend.shape[1], rend.shape[0]
print(f"Source : {SOURCE} — {N} titres, {T} semaines, du {rend.index[0]:%d/%m/%Y} au {rend.index[-1]:%d/%m/%Y}")

(prix[list(TITRES)].loc["2020":"2024"] / prix[list(TITRES)].loc["2020":"2024"].iloc[0]).plot(
    title="Évolution de 1 dollar investi début 2020", logy=True)
plt.show()

## 2. Rendements, variances, covariances

Notons $r_{i,t}$ le rendement hebdomadaire du titre $i$ la semaine $t$. Les estimateurs usuels sont :

$$\hat\mu_i = \frac{1}{T}\sum_{t=1}^{T} r_{i,t}, \qquad
\hat\sigma_{ij} = \frac{1}{T-1}\sum_{t=1}^{T} (r_{i,t}-\hat\mu_i)(r_{j,t}-\hat\mu_j)$$

Pour annualiser, on multiplie la moyenne et la covariance par 52 (et donc la volatilité par $\sqrt{52}$),
en supposant les rendements hebdomadaires indépendants d'une semaine à l'autre.

### Exercice 2.1
Calculez le vecteur des rendements moyens annualisés `mu`, la matrice de covariance annualisée `Sigma`
et le vecteur des volatilités annuelles `vol`.

In [ ]:
mu = rend.mean() * SEMAINES
Sigma = rend.cov() * SEMAINES
vol = pd.Series(np.sqrt(np.diag(Sigma)), index=Sigma.index)

stats = pd.DataFrame({"Secteur": pd.Series(TITRES), "Rendement moyen annuel": mu,
                      "Volatilité annuelle": vol})
stats

La matrice de covariance est difficile à lire telle quelle. On regarde plutôt la **matrice des corrélations**,
$\rho_{ij} = \sigma_{ij} / (\sigma_i \sigma_j)$, comprise entre −1 et 1.

In [ ]:
afficher_matrice(rend.corr(), "Corrélations des rendements hebdomadaires (2020-2024)")

### Question 2.2
Quelles paires de titres sont les plus corrélées ? Est-ce cohérent avec leurs secteurs ?
Combien de paramètres distincts a-t-on estimés pour décrire ces dix titres ?

In [ ]:
n_params = N + N + N * (N - 1) // 2
print(f"{N} rendements espérés + {N} variances + {N*(N-1)//2} covariances = {n_params} paramètres")
for n in (10, 100, 1000):
    print(f"  avec {n:>5} titres : {2*n + n*(n-1)//2:>9,} paramètres".replace(",", " "))

> **Corrigé — commentaire.** Les paires d'un même secteur (les deux banques, les deux pétrolières, les deux valeurs technologiques)
> ressortent nettement : elles partagent des risques communs au-delà du marché. Avec dix titres,
> on estime déjà 65 paramètres ; le nombre de covariances croît comme le carré du nombre de titres.
> C'est le « mur » que rencontre Markowitz dans les années 1950.

## 3. L'éléphant : estimer les rendements espérés

La moyenne historique est l'estimateur le plus naturel, mais aussi le plus fragile.
Son erreur-type (l'écart-type de l'estimateur) vaut, en annualisé :

$$\text{ET}(\hat\mu_i) \approx \frac{\sigma_i}{\sqrt{\text{nombre d'années}}}$$

Robert Merton l'a souligné en 1980 : observer plus souvent (chaque jour plutôt que chaque semaine)
n'améliore pas cette précision. Seule la **durée** de l'historique compte.

### Exercice 3.1
Calculez l'erreur-type de chaque rendement moyen et un intervalle de confiance à 95 %
($\hat\mu \pm 1{,}96 \times \text{ET}$).

In [ ]:
annees = T / SEMAINES
et_mu = vol / np.sqrt(annees)
incertitude = pd.DataFrame({"Rendement moyen": mu, "Erreur-type": et_mu,
                            "Borne basse 95 %": mu - 1.96 * et_mu, "Borne haute 95 %": mu + 1.96 * et_mu})
incertitude

> **Corrigé — commentaire.** Avec cinq ans de données et des volatilités de 20 à 35 %, l'erreur-type dépasse souvent 10 points :
> les intervalles de confiance contiennent presque tous zéro. On ne sait pas, statistiquement, si ces titres
> ont un rendement espéré de 0 % ou de 25 % par an.

### Rééchantillonner l'historique (bootstrap)

Pour visualiser cette incertitude, on tire au hasard, avec remise, 260 semaines parmi les 260 observées,
et on recalcule les moyennes. On recommence 2 000 fois : la dispersion obtenue montre ce qu'aurait pu
donner un autre historique, tout aussi plausible.

In [ ]:
rng = np.random.default_rng(0)
tirages = np.array([rend.values[rng.integers(0, T, T)].mean(axis=0) * SEMAINES for _ in range(2000)])

fig, ax = plt.subplots(figsize=(10, 5))
boites(ax, tirages, rend.columns)
ax.axhline(0, color="black", lw=0.8)
ax.set_title("Rendement moyen annuel : 2 000 historiques rééchantillonnés")
ax.set_ylabel("Rendement annuel")
plt.show()

### Le rétrécissement de Jorion (1986)

Philippe Jorion propose de **rétrécir** chaque moyenne vers une valeur commune $\mu_0$, le rendement
du portefeuille de variance minimale :

$$\hat\mu^{\text{BS}} = (1-\phi)\,\hat\mu + \phi\,\mu_0\,\mathbf{1}, \qquad
\phi = \frac{N+2}{(N+2) + T\,(\hat\mu-\mu_0\mathbf{1})^\top \hat\Sigma^{-1} (\hat\mu-\mu_0\mathbf{1})}$$

où $\hat\Sigma$ est la covariance empirique hebdomadaire, corrigée par le facteur $(T-1)/(T-N-2)$.
Plus l'historique est court, plus $\phi$ est grand, et plus on tire les moyennes vers $\mu_0$.

In [ ]:
def jorion(rend):
    """Estimateur bayésien de Jorion (1986) des rendements espérés, en annualisé."""
    T, N = rend.shape
    m = rend.mean().values
    S = rend.cov().values * (T - 1) / (T - N - 2)
    Si = np.linalg.inv(S)
    un = np.ones(N)
    mu0 = un @ Si @ m / (un @ Si @ un)
    ecart = m - mu0
    phi = (N + 2) / ((N + 2) + T * ecart @ Si @ ecart)
    return pd.Series((1 - phi) * m + phi * mu0, index=rend.columns) * SEMAINES, phi

mu_jorion, phi = jorion(rend)
print(f"Intensité de rétrécissement phi = {phi:.2f}")
pd.DataFrame({"Moyenne historique": mu, "Jorion": mu_jorion})

> **Pour la suite**, on garde la moyenne historique `mu`, la plus simple et la plus utilisée.
> Vous pouvez remplacer `mu` par `mu_jorion` et relancer le notebook pour comparer.

## 4. Résoudre le problème pas à pas

### 4.1 Avec deux titres

Avec deux titres A et B, de poids $w_A$ et $w_B = 1 - w_A$ :

$$\mu_p = w_A\,\mu_A + w_B\,\mu_B, \qquad
\sigma_p^2 = w_A^2\,\sigma_A^2 + w_B^2\,\sigma_B^2 + 2\,w_A w_B\,\sigma_{AB}$$

### Exercice 4.1
Pour Apple et JPMorgan, calculez rendement et volatilité du portefeuille pour $w_A$ allant de −0,5 à 1,5,
puis tracez la courbe obtenue.

In [ ]:
a, b = "AAPL", "JPM"
wA = np.linspace(-0.5, 1.5, 201)
wB = 1 - wA
mu_p = wA * mu[a] + wB * mu[b]
var_p = wA**2 * Sigma.loc[a, a] + wB**2 * Sigma.loc[b, b] + 2 * wA * wB * Sigma.loc[a, b]
sigma_p = np.sqrt(var_p)

plt.plot(sigma_p, mu_p, label="Portefeuilles A + B")
plt.scatter([vol[a], vol[b]], [mu[a], mu[b]], color="black", zorder=3)
for t in (a, b):
    plt.annotate(t, (vol[t], mu[t]), textcoords="offset points", xytext=(6, 4))
i_min = sigma_p.argmin()
plt.scatter(sigma_p[i_min], mu_p[i_min], color="C3", zorder=3, label=f"Variance minimale (w_A = {wA[i_min]:.2f})")
plt.xlabel("Volatilité annuelle"); plt.ylabel("Rendement espéré annuel")
plt.title(f"Deux titres : corrélation {rend[a].corr(rend[b]):.2f}"); plt.legend(); plt.show()

> **Corrigé — commentaire.** La courbe est une hyperbole. Son point le plus à gauche, le portefeuille de variance minimale,
> est moins risqué que chacun des deux titres : c'est l'effet de la diversification, d'autant plus fort
> que la corrélation est faible.

### 4.2 Avec N titres : la formule fermée

On cherche les poids $w$ qui minimisent la variance pour un rendement cible $m$, en investissant tout le capital :

$$\min_{w} \; \tfrac{1}{2}\, w^\top \Sigma w \quad \text{sous} \quad w^\top \mu = m, \qquad w^\top \mathbf{1} = 1$$

Le lagrangien $\mathcal{L} = \tfrac{1}{2} w^\top \Sigma w - \lambda (w^\top\mu - m) - \gamma (w^\top \mathbf{1} - 1)$
donne, en annulant sa dérivée :

$$w^* = \Sigma^{-1}(\lambda\,\mu + \gamma\,\mathbf{1})$$

En posant $A = \mathbf{1}^\top\Sigma^{-1}\mathbf{1}$, $B = \mathbf{1}^\top\Sigma^{-1}\mu$,
$C = \mu^\top\Sigma^{-1}\mu$ et $D = AC - B^2$, les contraintes donnent :

$$\lambda = \frac{Am - B}{D}, \qquad \gamma = \frac{C - Bm}{D}, \qquad \sigma^2(m) = \frac{Am^2 - 2Bm + C}{D}$$

### Exercice 4.2
Calculez $A$, $B$, $C$, $D$, puis écrivez une fonction `portefeuille_cible(m)` qui renvoie $w^*$.
Vérifiez que les deux contraintes sont respectées et que la variance obtenue vaut bien $\sigma^2(m)$.

In [ ]:
Si = np.linalg.inv(Sigma.values)
un = np.ones(N)
m_vec = mu.values

A = un @ Si @ un
B = un @ Si @ m_vec
C = m_vec @ Si @ m_vec
D = A * C - B**2

def portefeuille_cible(m):
    lam = (A * m - B) / D
    gam = (C - B * m) / D
    return Si @ (lam * m_vec + gam * un)

m_test = 0.12
w = portefeuille_cible(m_test)
print("Poids :", pd.Series(w, index=TITRES).round(3).to_dict())
print(f"Somme des poids : {w.sum():.6f}   Rendement : {w @ m_vec:.6f}")
print(f"Variance : {w @ Sigma.values @ w:.6f}   Formule : {(A*m_test**2 - 2*B*m_test + C)/D:.6f}")

### 4.3 Deux portefeuilles remarquables

- Le **portefeuille de variance minimale** (GMV) : $w_{\text{GMV}} = \Sigma^{-1}\mathbf{1} / A$.
  Il ne dépend pas des rendements espérés.
- Le **portefeuille tangent**, qui maximise le ratio de Sharpe $(\mu_p - r_f)/\sigma_p$ :
  $w_{\text{T}} = \Sigma^{-1}(\mu - r_f\mathbf{1}) \,/\, \mathbf{1}^\top\Sigma^{-1}(\mu - r_f\mathbf{1})$.
  Combiné à l'actif sans risque, il forme la droite de marché des capitaux.

In [ ]:
def tangent(Sigma_, mu_, rf=TAUX_SANS_RISQUE):
    x = np.linalg.solve(Sigma_, mu_ - rf)
    return x / x.sum()

w_gmv = Si @ un / A
w_tan = tangent(Sigma.values, m_vec)
print(f"Rendement du portefeuille de variance minimale : {B / A:.1%} (taux sans risque : {TAUX_SANS_RISQUE:.0%})")
if B / A < TAUX_SANS_RISQUE + 0.01:
    print("⚠️  Ce rendement est proche du taux sans risque : le portefeuille tangent devient instable ou mal défini.")

def caract(w_):
    r, s = w_ @ m_vec, np.sqrt(w_ @ Sigma.values @ w_)
    return r, s, (r - TAUX_SANS_RISQUE) / s

r_t, s_t, sh_t = caract(w_tan)
m_haut = max(m_vec.max() + 0.10, r_t + 0.05) if r_t > B / A else m_vec.max() + 0.10
m_grille = np.linspace(min(B / A, m_vec.min()) - 0.05, m_haut, 300)
sig_grille = np.sqrt((A * m_grille**2 - 2 * B * m_grille + C) / D)
plt.plot(sig_grille, m_grille, label="Frontière efficiente (vente à découvert permise)")
plt.scatter(vol, mu, color="black", s=15, zorder=3)
for t in TITRES:
    plt.annotate(t, (vol[t], mu[t]), textcoords="offset points", xytext=(4, 3), fontsize=8)
plt.scatter(*caract(w_gmv)[1::-1], color="C2", s=60, zorder=4, label="Variance minimale")
plt.scatter(s_t, r_t, color="C3", marker="*", s=200, zorder=4, label=f"Tangent (Sharpe {sh_t:.2f})")
x = np.linspace(0, sig_grille.max(), 50)
plt.plot(x, TAUX_SANS_RISQUE + sh_t * x, "--", color="C3", lw=1, label="Droite de marché des capitaux")
plt.xlabel("Volatilité annuelle"); plt.ylabel("Rendement espéré annuel"); plt.xlim(0); plt.legend(fontsize=8)
plt.title("Frontière efficiente de Markowitz"); plt.show()

pd.DataFrame({"Variance minimale": w_gmv, "Tangent": w_tan}, index=TITRES).style.format("{:.1%}")

> **Corrigé — commentaire.** Sans contrainte, le portefeuille tangent prend souvent des positions extrêmes, avec des ventes à découvert
> importantes. C'est un premier signe de la fragilité de l'optimisation, que l'on étudie à la section 6.

### 4.4 Le théorème de séparation de Tobin

Avec un actif sans risque, chaque investisseur combine le portefeuille tangent et l'actif sans risque.
Un investisseur d'aversion au risque $\gamma$ qui maximise $\mu_p - \frac{\gamma}{2}\sigma_p^2$ place
dans le portefeuille tangent la fraction :

$$y^* = \frac{\mu_T - r_f}{\gamma\, \sigma_T^2}$$

et le reste, $1 - y^*$, au taux sans risque (si $y^* > 1$, il emprunte). La **composition** de la partie
risquée est la même pour tous : seul le dosage change. C'est le théorème de séparation (chapitre 1, partie 6).

### Exercice 4.4
Pour trois investisseurs d'aversion $\gamma = 4$, $10$ et $25$, calculez $y^*$, puis le rendement espéré et la
volatilité de leur position. Vérifiez que les trois points sont sur la droite de marché des capitaux, et que
les poids de chaque titre **rapportés à la partie risquée** sont identiques pour les trois.

In [ ]:
lignes = {}
for gamma in [4, 10, 25]:
    y = (r_t - TAUX_SANS_RISQUE) / (gamma * s_t**2)
    w_total = y * w_tan                      # poids de chaque titre dans le patrimoine total
    r_p = TAUX_SANS_RISQUE + y * (r_t - TAUX_SANS_RISQUE)
    s_p = abs(y) * s_t
    lignes[f"γ = {gamma}"] = {"Part dans le tangent": y, "Part sans risque": 1 - y,
                              "Rendement espéré": r_p, "Volatilité": s_p,
                              "Écart à la CML": r_p - (TAUX_SANS_RISQUE + sh_t * s_p),
                              f"Poids de {list(TITRES)[0]} / partie risquée": w_total[0] / w_total.sum()}
pd.DataFrame(lignes).style.format("{:.2%}")

> **Corrigé — commentaire.** L'écart à la droite de marché des capitaux est nul pour les trois investisseurs, et la dernière ligne est
> identique : la partie risquée a toujours la composition du portefeuille tangent. Un investisseur prudent
> (γ = 25) détient surtout de l'actif sans risque ; un investisseur audacieux (γ = 4) peut emprunter pour
> détenir plus de 100 % de portefeuille tangent. Si les chiffres paraissent extrêmes, c'est que le portefeuille
> tangent estimé l'est lui-même (section 7). C'est ce résultat qui, au chapitre 4, conduit au CAPM.

## 5. Résolution numérique, avec ou sans vente à découvert

En pratique, on interdit souvent la vente à découvert ($w_i \ge 0$). La formule fermée ne s'applique plus :
il faut un algorithme itératif. Markowitz proposait dès 1956 l'algorithme de la ligne critique ;
on utilise ici l'optimiseur générique SLSQP de `scipy`.

### Exercice 5.1
Vérifiez d'abord que l'optimiseur retrouve la formule fermée lorsque les poids négatifs sont permis.

In [ ]:
def min_variance(Sigma_, mu_, cible, positifs=False):
    n = len(mu_)
    contraintes = [{"type": "eq", "fun": lambda w: w.sum() - 1},
                   {"type": "eq", "fun": lambda w: w @ mu_ - cible}]
    res = minimize(lambda w: w @ Sigma_ @ w, np.ones(n) / n, jac=lambda w: 2 * Sigma_ @ w,
                   constraints=contraintes, bounds=[(0, 1)] * n if positifs else None,
                   method="SLSQP", options={"ftol": 1e-12, "maxiter": 1000})
    return res.x

w_num = min_variance(Sigma.values, m_vec, m_test)
print("Écart maximal entre optimiseur et formule fermée :", np.abs(w_num - portefeuille_cible(m_test)).max())

### Exercice 5.2 — La frontière sans vente à découvert

In [ ]:
cibles = np.linspace(m_vec.min(), m_vec.max(), 40)
frontiere_pos = np.array([min_variance(Sigma.values, m_vec, c, positifs=True) for c in cibles])
sig_pos = np.sqrt(np.einsum("ij,jk,ik->i", frontiere_pos, Sigma.values, frontiere_pos))

plt.plot(sig_grille, m_grille, label="Vente à découvert permise")
plt.plot(sig_pos, cibles, label="Poids positifs uniquement", lw=2.5)
plt.scatter(vol, mu, color="black", s=15, zorder=3)
plt.xlabel("Volatilité annuelle"); plt.ylabel("Rendement espéré annuel"); plt.xlim(0); plt.legend()
plt.title("La contrainte de positivité réduit la frontière"); plt.show()

plt.stackplot(cibles, frontiere_pos.T, labels=list(TITRES))
plt.xlabel("Rendement cible"); plt.ylabel("Poids"); plt.title("Composition le long de la frontière (poids positifs)")
plt.legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc="upper left"); plt.ylim(0, 1); plt.show()

> **Corrigé — commentaire.** Sans vente à découvert, la frontière est plus courte et légèrement à droite : on perd des possibilités.
> Le graphique empilé montre que la plupart des portefeuilles efficients ne contiennent que quelques titres :
> l'optimiseur concentre, il ne diversifie pas spontanément.

## 6. Le mur du calcul

Inverser une matrice $N \times N$ demande de l'ordre de $N^3$ opérations. En 1963, William Sharpe indiquait
qu'une analyse de 100 titres prenait environ **33 minutes** sur un IBM 7090, et que le code standard ne pouvait
pas dépasser **249 titres**. Mesurons ce que coûte aujourd'hui le même calcul.

On fabrique pour cela des matrices de covariance réalistes de grande taille, à partir d'un modèle à facteurs.

In [ ]:
def covariance_aleatoire(n, rng):
    """Matrice de covariance annuelle plausible : 3 facteurs communs + risque propre."""
    Bf = rng.normal(1, 0.3, (n, 3)) * np.array([0.15, 0.08, 0.05])
    return Bf @ Bf.T + np.diag(rng.uniform(0.02, 0.09, n))

def chrono(f, repetitions=3):
    t0 = time.perf_counter()
    for _ in range(repetitions):
        f()
    return (time.perf_counter() - t0) / repetitions

rng = np.random.default_rng(1)
lignes = []
for n in (10, 50, 100, 250, 500, 1000, 2000):
    S_n = covariance_aleatoire(n, rng)
    mu_n = rng.uniform(0.04, 0.12, n)
    t_inv = chrono(lambda: np.linalg.inv(S_n))
    t_slsqp = chrono(lambda: min_variance(S_n, mu_n, mu_n.mean(), positifs=True), 1) if n <= 100 else np.nan
    lignes.append({"Titres": n, "Paramètres à estimer": 2 * n + n * (n - 1) // 2,
                   "Inversion (ms)": 1000 * t_inv, "Optimisation sans vente à découvert (s)": t_slsqp})
cout = pd.DataFrame(lignes).set_index("Titres")
cout.style.format({"Paramètres à estimer": "{:,.0f}", "Inversion (ms)": "{:.2f}",
                   "Optimisation sans vente à découvert (s)": "{:.2f}"}, na_rep="—")

In [ ]:
plt.loglog(cout.index, cout["Inversion (ms)"], "o-", label="Temps mesuré")
ref = cout["Inversion (ms)"].iloc[-1] * (cout.index / cout.index[-1]) ** 3
plt.loglog(cout.index, ref, "--", color="grey", label="Croissance en N³")
plt.xlabel("Nombre de titres"); plt.ylabel("Temps d'inversion (ms)"); plt.legend()
plt.title("Le coût de l'inversion croît comme le cube du nombre de titres"); plt.show()

> **Corrigé — commentaire.** Un ordinateur portable inverse une matrice de 1 000 titres en quelques dizaines de millisecondes.
> L'argument du temps de calcul, décisif en 1963, a disparu. Pour les petites tailles, la croissance mesurée
> est plus lente que N³ car les bibliothèques modernes sont dominées par des coûts fixes ; elle rejoint
> la pente théorique pour les grandes matrices. En revanche, le nombre de paramètres à estimer, lui,
> continue de croître comme N² : le vrai problème a changé de nature.

## 7. Sensibilité : l'optimiseur, « maximiseur d'erreurs »

En 1989, Richard Michaud qualifie les optimiseurs moyenne-variance de « maximiseurs d'erreurs » :
ils surpondèrent les titres dont le rendement est surestimé et le risque sous-estimé.

### Exercice 7.1 — Un point de rendement
Augmentez de 1 point le rendement espéré de JPMorgan, recalculez le portefeuille tangent et comparez les poids.

In [ ]:
mu_choc = mu.copy()
mu_choc["JPM"] += 0.01
w_avant = tangent(Sigma.values, mu.values)
w_apres = tangent(Sigma.values, mu_choc.values)

choc = pd.DataFrame({"Avant": w_avant, "Après +1 pt sur JPM": w_apres, "Écart": w_apres - w_avant}, index=TITRES)
print(f"Somme des écarts absolus de poids : {np.abs(w_apres - w_avant).sum():.0%}")
choc.style.format("{:+.1%}")

In [ ]:
choc[["Avant", "Après +1 pt sur JPM"]].plot.bar(title="Portefeuille tangent : effet d'un point de rendement sur JPM")
plt.axhline(0, color="black", lw=0.8); plt.ylabel("Poids"); plt.show()

> **Corrigé — commentaire.** Un seul point de rendement espéré, bien en deçà de l'erreur-type calculée à la section 3, déplace
> nettement les poids, d'autant plus que les titres sont corrélés entre eux. Regardez en particulier l'autre banque, BAC : très corrélée à JPM,
> elle est souvent vendue pour financer l'achat de JPM. L'optimiseur croit voir un arbitrage entre deux
> titres jumeaux, alors que l'écart de rendement n'est que du bruit d'estimation.

### Exercice 7.2 — Des erreurs plausibles
Ajoutez maintenant à **tous** les rendements espérés une erreur aléatoire de l'ordre de leur erreur-type
(tirée selon une loi normale), 1 000 fois, et observez la dispersion des poids.

Avec vente à découvert, on suit un investisseur dont l'aversion au risque $\gamma$ est fixe :
ses poids valent $w = \Sigma^{-1}(\mu - r_f)/\gamma$, et la part non investie en actions est placée
au taux sans risque. On choisit $\gamma$ pour qu'il soit investi à 100 % en actions avec les paramètres
estimés. Cela évite la renormalisation du portefeuille tangent, qui explose quand la somme des poids
s'approche de zéro. Sans vente à découvert, on recalcule le portefeuille tangent (200 tirages,
l'optimisation est plus lente).

In [ ]:
rng = np.random.default_rng(2)
bruits = rng.normal(0, et_mu.values, (1000, N))
gamma = np.linalg.solve(Sigma.values, mu.values - TAUX_SANS_RISQUE).sum()   # investi à 100 % au départ
poids_mc = np.array([np.linalg.solve(Sigma.values, mu.values + e - TAUX_SANS_RISQUE) / gamma for e in bruits])

def tangent_positif(Sigma_, mu_, rf=TAUX_SANS_RISQUE):
    n = len(mu_)
    res = minimize(lambda w: -(w @ mu_ - rf) / np.sqrt(w @ Sigma_ @ w), np.ones(n) / n,
                   constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}],
                   bounds=[(0, 1)] * n, method="SLSQP", options={"ftol": 1e-10, "maxiter": 500})
    return res.x

poids_mc_pos = np.array([tangent_positif(Sigma.values, mu.values + e) for e in bruits[:200]])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
boites(axes[0], np.clip(poids_mc, -3, 3), list(TITRES))
axes[0].set_title("Aversion au risque fixe, vente à découvert permise (bornés à ±300 %)")
boites(axes[1], poids_mc_pos, list(TITRES))
axes[1].set_title("Tangent, poids positifs")
for ax in axes:
    ax.axhline(0, color="black", lw=0.8)
plt.show()

presence = pd.Series((poids_mc_pos > 0.01).mean(axis=0), index=TITRES)
print("Fréquence de présence de chaque titre dans le portefeuille tangent à poids positifs :")
print(presence.map("{:.0%}".format).to_string())

> **Corrigé — commentaire.** Avec des erreurs parfaitement plausibles, les poids changent du tout au tout, et parfois de signe.
> Sans vente à découvert, l'instabilité prend une autre forme : les titres entrent et sortent du portefeuille
> au gré du bruit. C'est le diagnostic de Michaud. Le notebook 2 montre comment le modèle de Sharpe
> atténue une partie du problème, celle qui vient de la matrice de covariance.

## 8. Bilan

| Étape | Ce qu'on a vu |
|---|---|
| Données | 10 titres demandent déjà 65 paramètres ; 1 000 titres en demanderaient plus de 500 000. |
| Rendements espérés | L'erreur-type de la moyenne historique dépasse souvent 10 points par an. |
| Formule fermée | Tout passe par l'inverse de la matrice de covariance. |
| Calcul | Ce qui coûtait 33 minutes en 1963 prend aujourd'hui quelques millisecondes. |
| Sensibilité | Un point d'erreur sur un rendement bouleverse le portefeuille optimal. |

**Pour aller plus loin**

1. Refaites la section 7 avec `mu_jorion` à la place de `mu`. L'instabilité diminue-t-elle ?
2. Estimez les paramètres sur 2015-2019 au lieu de 2020-2024. Le portefeuille tangent ressemble-t-il au précédent ?
3. Ajoutez une contrainte de poids maximal de 25 % par titre. Que devient la frontière ?

**Suite** : *Notebook 2 — Le modèle diagonal de Sharpe.*